In [0]:
# HIGH_GARDEN_CATALOG_PARAMETER
dbutils.widgets.text("catalog", "high_garden")
catalog = dbutils.widgets.get("catalog").strip() or "high_garden"
print(f"Using Unity Catalog: {catalog}")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_TABLE = f"{catalog}.silver.coffee_consumption"
TARGET_TABLE = f"{catalog}.gold.forecasting_features"

df = spark.table(SOURCE_TABLE)

print("Rows:", df.count())
display(df.limit(10))

In [0]:
time_window = (
    Window
    .partitionBy(
        "country",
        "coffee_type"
    )
    .orderBy(
        "start_year"
    )
)

In [0]:
features_df = (
    df
    .withColumn(
        "lag_1",
        F.lag(
            "domestic_consumption", 1
        ).over(time_window)
    )
    .withColumn(
        "lag_2",
        F.lag(
            "domestic_consumption", 2
        ).over(time_window)
    )
    .withColumn(
        "lag_3",
        F.lag(
            "domestic_consumption", 3
        ).over(time_window)
    )
    .withColumn(
        "lag_5",
        F.lag(
            "domestic_consumption", 5
        ).over(time_window)
    )
)

In [0]:
rolling_3 = (
    time_window
    .rowsBetween(-3, -1)
)

rolling_5 = (
    time_window
    .rowsBetween(-5, -1)
)

In [0]:
features_df = (
    features_df
    .withColumn(
        "rolling_mean_3",
        F.avg(
            "domestic_consumption"
        ).over(rolling_3)
    )
    .withColumn(
        "rolling_mean_5",
        F.avg(
            "domestic_consumption"
        ).over(rolling_5)
    )
)

In [0]:
features_df = (
    features_df
    .withColumn(
        "rolling_std_3",
        F.stddev_samp(
            "domestic_consumption"
        ).over(rolling_3)
    )
)

In [0]:
features_df = (
    features_df
    .withColumn(
        "historical_growth_1y",
        F.when(
            F.col("lag_2") > 0,
            (
                F.col("lag_1")
                - F.col("lag_2")
            )
            / F.col("lag_2")
        )
    )
)

In [0]:
features_df = (
    features_df
    .withColumn(
        "lag_1_zero",
        (
            F.col("lag_1") == 0
        ).cast("int")
    )
)

In [0]:
features_df = (
    features_df
    .withColumnRenamed(
        "domestic_consumption",
        "target"
    )
)

In [0]:
series_stats = (
    df
    .groupBy(
        "country",
        "coffee_type"
    )
    .agg(
        F.max(
            "domestic_consumption"
        ).alias("series_max")
    )
)

In [0]:
features_df = (
    features_df
    .join(
        series_stats,
        on=[
            "country",
            "coffee_type"
        ],
        how="left"
    )
    .withColumn(
        "ml_eligible",
        F.col("series_max") > 0
    )
)

In [0]:
features_df = features_df.select(
    "country",
    "coffee_type",
    "crop_year",
    "start_year",
    "end_year",

    "lag_1",
    "lag_2",
    "lag_3",
    "lag_5",

    "rolling_mean_3",
    "rolling_mean_5",
    "rolling_std_3",

    "historical_growth_1y",
    "lag_1_zero",

    "target",
    "zero_flag",

    "series_max",
    "ml_eligible"
)

In [0]:
display(
    features_df
    .filter(
        F.col("country") == "Brazil"
    )
    .orderBy("start_year")
)

In [0]:
training_ready_df = (
    features_df
    .filter(
        F.col("ml_eligible") == True
    )
    .filter(
        F.col("lag_1").isNotNull()
        &
        F.col("lag_2").isNotNull()
        &
        F.col("lag_3").isNotNull()
        &
        F.col("rolling_mean_3").isNotNull()
    )
)

In [0]:
print(
    "Feature rows:",
    features_df.count()
)

print(
    "Training-ready rows:",
    training_ready_df.count()
)

In [0]:
leakage_check = (
    training_ready_df
    .filter(
        F.col("lag_1").isNull()
        |
        F.col("lag_2").isNull()
        |
        F.col("lag_3").isNull()
    )
    .count()
)

assert leakage_check == 0

print(
    "Temporal feature validation passed."
)

In [0]:
assert (
    training_ready_df
    .filter(
        F.col("target").isNull()
    )
    .count()
    == 0
), "Null targets found"

assert (
    training_ready_df
    .filter(
        F.col("target") < 0
    )
    .count()
    == 0
), "Negative targets found"

print(
    "Target validation passed."
)

In [0]:
(
    features_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

In [0]:
display(spark.sql(f"""
SELECT *
FROM {catalog}.gold.forecasting_features
ORDER BY country, start_year
LIMIT 50;
"""))


In [0]:
display(spark.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT country) AS countries,
    SUM(
        CASE
            WHEN ml_eligible THEN 1
            ELSE 0
        END
    ) AS eligible_rows
FROM {catalog}.gold.forecasting_features;
"""))
